# Assignment 1 - Part 2: Comparing pandas vs polars

## Introduction

In each implementation I add the following two technical indicators:

- 20-Day Simple Moving Average (SMA_20) of Close price
  - Uses the average closing price of the previous 20 days to try and predict the current days closing price.
- Relative Strength Index (RSI)
  - Measures the stock's recent price momentum on a 0-100 scale. Compares the average gains and losses of the stock over the previous 14 days, computes the ratio of these average gains and losses, then normalizes the ratio to a scale of 0-100
    - $RSI = 100 - \left(\frac{100}{1 + RS}\right)$, Where $RS = \frac{avg \text{ gain}}{avg \text{ loss}}$

### Splitting the Data

Since this is time series data and the data needs to remain grouped by stock name, I wrote two functions (one for pandas and one for polars) to split the data into training and testing sets. The functions group by stock name and iterate over each company, selecting the first 80% of rows to be in the training set and the remaining rows to be in the test set. This ensures that the oldest dates are in the training data and the more recent dates are in the testing set.

### Models

I use **Random Forest Regressor** and **XGBOOST** to predict the next day stock prices for each company.


## Pandas Implementation

In [18]:
import pandas as pd

df_pd = pd.read_csv('all_stocks_5yr.csv', parse_dates=['date'])
df_pd = df_pd.sort_values(['name', 'date'])

df_pd.info()


<class 'pandas.DataFrame'>
Index: 619040 entries, 71611 to 619039
Data columns (total 7 columns):
 #   Column  Non-Null Count   Dtype         
---  ------  --------------   -----         
 0   date    619040 non-null  datetime64[us]
 1   open    619029 non-null  float64       
 2   high    619032 non-null  float64       
 3   low     619032 non-null  float64       
 4   close   619040 non-null  float64       
 5   volume  619040 non-null  int64         
 6   name    619040 non-null  str           
dtypes: datetime64[us](1), float64(4), int64(1), str(1)
memory usage: 39.7 MB


Adding technical indicators

In [26]:
# SMA: 20-day moving average of close
df_pd['SMA_20'] = df_pd.groupby('name')['close'].transform(lambda x: x.rolling(20).mean())

# RSI: 14-day relative strength index
delta = df_pd.groupby('name')['close'].transform(lambda x: x.diff())
gain = delta.where(delta > 0, 0).rolling(14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
rs = gain / loss
df_pd['RSI_14'] = 100 - (100 / (1 + rs))

df_pd.head(25)

,date,open,high,low,close,volume,name,SMA_20,RSI_14
71611,2013-02-08,45.07,45.350,45.000,45.08,1824755,A,NaN,NaN
71612,2013-02-11,45.17,45.180,44.450,44.60,2915405,A,NaN,NaN
71613,2013-02-12,44.81,44.950,44.500,44.62,2373731,A,NaN,NaN
71614,2013-02-13,44.81,45.240,44.680,44.75,2052338,A,NaN,NaN
71615,2013-02-14,44.72,44.780,44.360,44.58,3826245,A,NaN,NaN
71616,2013-02-15,43.48,44.240,42.210,42.25,14657315,A,NaN,NaN
71617,2013-02-19,42.21,43.120,42.210,43.01,4116141,A,NaN,NaN
71618,2013-02-20,42.84,42.850,42.225,42.24,3873183,A,NaN,NaN
71619,2013-02-21,42.14,42.140,41.470,41.63,3415149,A,NaN,NaN
71620,2013-02-22,41.83,42.070,41.580,41.80,3354862,A,NaN,NaN


Splitting the data

In [15]:
def split_train_test_pd(df_pd, test_size=0.2):
    train_dfs = []
    test_dfs = []
    for name, group in df_pd.groupby('name'):
        n = len(group)
        train_split = int(n * (1 - test_size))
        train = group.iloc[:train_split]
        test = group.iloc[train_split:]
        train_dfs.append(train)
        test_dfs.append(test)
    return pd.concat(train_dfs), pd.concat(test_dfs)


In [17]:
df_pd_train, df_pd_test = split_train_test_pd(df_pd)
print(f"Train: {len(df_pd_train)}, Test: {len(df_pd_test)}")
print("Train date range:", df_pd_train['date'].min(), "to", df_pd_train['date'].max())
print("Test date range:", df_pd_test['date'].min(), "to", df_pd_test['date'].max())


Train: 495125, Test: 123915
Train date range: 2013-02-08 00:00:00 to 2018-01-25 00:00:00
Test date range: 2017-02-08 00:00:00 to 2018-02-07 00:00:00


### Models

## Polars Implementation

In [22]:
import polars as pl

df_pl = pl.read_csv('all_stocks_5yr.csv')
df_pl = df_pl.with_columns(pl.col('date').str.to_datetime())
df_pl = df_pl.sort(['name', 'date'])


Adding technical indicators

In [ ]:
# SMA: 20-day moving average of close
df_pl = df_pl.with_columns(
    pl.col('close').rolling_mean(20).over('name').alias('SMA_20')
)

# RSI: 14-day relative strength index
df_pl = df_pl.with_columns(
    pl.col('close').diff().over('name').alias('delta')
)
df_pl = df_pl.with_columns([
    pl.col('delta').clip(lower_bound=0).rolling_mean(14).over('name').alias('avg_gain'),
    (-pl.col('delta').clip(upper_bound=0)).rolling_mean(14).over('name').alias('avg_loss')
])
df_pl = df_pl.with_columns([
    (100 - (100 / (1 + (pl.col('avg_gain') / pl.col('avg_loss'))))).alias('RSI_14')
])

# dropping temp columns
df_pl = df_pl.drop(['delta', 'avg_gain', 'avg_loss'])

print(df_pl.select(['name', 'date', 'close', 'SMA_20', 'RSI_14']).head(25))


shape: (20, 5)
┌──────┬─────────────────────┬───────┬─────────┬───────────┐
│ name ┆ date                ┆ close ┆ SMA_20  ┆ RSI_14    │
│ ---  ┆ ---                 ┆ ---   ┆ ---     ┆ ---       │
│ str  ┆ datetime[μs]        ┆ f64   ┆ f64     ┆ f64       │
╞══════╪═════════════════════╪═══════╪═════════╪═══════════╡
│ A    ┆ 2013-02-08 00:00:00 ┆ 45.08 ┆ null    ┆ null      │
│ A    ┆ 2013-02-11 00:00:00 ┆ 44.6  ┆ null    ┆ null      │
│ A    ┆ 2013-02-12 00:00:00 ┆ 44.62 ┆ null    ┆ null      │
│ A    ┆ 2013-02-13 00:00:00 ┆ 44.75 ┆ null    ┆ null      │
│ A    ┆ 2013-02-14 00:00:00 ┆ 44.58 ┆ null    ┆ null      │
│ …    ┆ …                   ┆ …     ┆ …       ┆ …         │
│ A    ┆ 2013-03-04 00:00:00 ┆ 42.03 ┆ null    ┆ 32.517007 │
│ A    ┆ 2013-03-05 00:00:00 ┆ 42.66 ┆ null    ┆ 37.688442 │
│ A    ┆ 2013-03-06 00:00:00 ┆ 43.24 ┆ null    ┆ 41.022592 │
│ A    ┆ 2013-03-07 00:00:00 ┆ 43.25 ┆ null    ┆ 41.939394 │
│ A    ┆ 2013-03-08 00:00:00 ┆ 43.03 ┆ 42.8085 ┆ 56.351792 │
└──────┴─

Splitting Data

In [19]:
def split_train_test_pl(df_pl, test_size=0.2):
    train_dfs = []
    test_dfs = []
    for name in df_pl['name'].unique():
        group = df_pl.filter(pl.col('name') == name)
        n_rows = group.height
        train_rows = int(n_rows * (1 - test_size))
        train = group.slice(0, train_rows)
        test = group.slice(train_rows, n_rows - train_rows)
        train_dfs.append(train)
        test_dfs.append(test)
    return pl.concat(train_dfs), pl.concat(test_dfs)


In [27]:
df_pl_train, df_pl_test = split_train_test_pl(df_pl)
print(f"Train: {df_pl_train.height}, Test: {df_pl_test.height}")


Train: 495125, Test: 123915
